In [ ]:
# Ch_15_TypedDict 예제 / @typing.overload 예제 / XML생성기
# More about TypeHint

In [ ]:
# typing.overload

''' 
typing.overload는 하나의 함수/메서드가 여러 시그니처(입력 타입 조합)를 가질 수 있음을 타입 체커(mypy 등)에게 알려주는 데 사용합니다. 
실제 실행되는 구현체는 하나이며, @overload로 장식된 함수들은 타입 힌트 제공용일 뿐 런타임에는 무시됩니다.
'''

from typing import overload, List, Union


class Calculator:
    """다양한 타입의 값을 합치는 가상의 계산기 클래스"""

    # 1. 정수 두 개를 더하면 정수를 반환
    @overload
    def sum(self, a: int, b: int) -> int: ...   
    # @overload 데코레이터	실제 로직 없이 시그니처(타입 조합)만 선언. 본문은 보통 ... (Ellipsis)로 채움

    # 2. 실수 두 개를 더하면 실수를 반환
    @overload
    def sum(self, a: float, b: float) -> float: ...

    # 3. 문자열 두 개를 합치면 문자열을 반환 (문자열 연결)
    @overload
    def sum(self, a: str, b: str) -> str: ...

    # 4. 리스트 하나를 넘기면 리스트 내부 요소들의 합을 반환
    @overload
    def sum(self, a: List[Union[int, float]]) -> Union[int, float]: ...

    # 실제 구현부 (실행 시 호출되는 유일한 코드)
    def sum(self, a, b=None):
        if b is None:
            # 리스트가 들어온 경우
            if isinstance(a, list):
                return __builtins__['sum'](a) if isinstance(__builtins__, dict) else sum(a)
            raise TypeError("지원하지 않는 타입입니다.")

        if isinstance(a, str) and isinstance(b, str):
            return a + b  # 문자열 연결
        if isinstance(a, (int, float)) and isinstance(b, (int, float)):
            return a + b  # 숫자 덧셈

        raise TypeError(f"지원하지 않는 타입 조합입니다: {type(a)}, {type(b)}")

In [4]:

# ===== 사용 예시 =====
calc = Calculator()

print(calc.sum(1, 2))          # 3            (int + int)
print(calc.sum(1.5, 2.5))      # 4.0          (float + float)
print(calc.sum("Hello, ", "World!"))  # Hello, World!  (str + str)
print(calc.sum([1, 2, 3, 4]))  # 10           (list 합산)

3
4.0
Hello, World!
10


In [ ]:
# TypedDict
'''
TypeDict는 일반적인 클래스와 완전히 다르다.

- 일반 클래스 : 객체(Object)를 생성하며, 데이터(Attribute)와 행위(Method)를 함께 묶어서 관리하기
             위해 사용함.
- TypeDict : 클래스의 문법을 빌려쓰기는 하지만, 실제로는 "이 딕셔너리는 어떤 Key와 어떤 Type을 
             가져와야 한다는 것을 정적 타입 검사기(mypy, pyright 등)에게 알려주기 위한 타입 힌트


만일 데이터(속성)와 메서드(기능)을 모두 가져야 한다면,
TypedDict 대신 dataclass를 사용해야 한다. (260811)

'''

from typing import TypedDict

class BookDict(TypedDict):
    isbn: str
    title: str
    authors: list[str]
    pagecount: int

pp = BookDict(
        title='Programming Pearls',
        authors='Jon Bentley',
        isbn='0201657880',
        pagecount=256
        )

In [ ]:
type(pp)    # Dict 형태 -> 클래스에서 생성된 인스턴스가 아니다. 즉 자료 구조

dict

In [7]:
pp.title  # 당연히 불가능하다

AttributeError: 'dict' object has no attribute 'title'

In [ ]:
pp['title']  # dict 형태라는 것을 확인

'Programming Pearls'

In [ ]:
BookDict.__annotations__    # Type Hint를 확인한다. (260811)

{'isbn': str, 'title': str, 'authors': list[str], 'pagecount': int}

In [11]:

AUTHOR_ELEMENT = '<AUTHOR>{}</AUTHOR>'

def to_xml(book: BookDict) -> str:
    elements: list[str] = []
    for key, value in book.items():
        if isinstance(value, list):
            elements.extend(
               AUTHOR_ELEMENT.format(n) for n in value)
        else:
            tag = key.upper()
            elements.append(f'<{tag}>{value}</{tag}>')
    xml = '\n\t'.join(elements)
    return f'<BOOK>\n\t{xml}\n</BOOK>'

xml_aa = to_xml(pp)
print(xml_aa)


<BOOK>
	<TITLE>Programming Pearls</TITLE>
	<AUTHORS>Jon Bentley</AUTHORS>
	<ISBN>0201657880</ISBN>
	<PAGECOUNT>256</PAGECOUNT>
</BOOK>


In [12]:
'''
pp : BookDict = { 'title': ..., ... }

pp (변수 이름): 우리가 데이터를 담을 변수의 이름입니다.
: BookDict (타입 어노테이션): **"이 pp라는 변수에는 BookDict라는 규칙(형태)을 가진 데이터만 들어와야 해!"**라고 선언하는 부분입니다.
= { ... } (값 할당): 실제로 변수에 값을 집어넣는 부분입니다. 여기서는 딕셔너리(dict)를 할당하고 있습니다.
'''

# 딕셔너리 구조를 유지하며 기능을 쓰고 싶은 경우 -> 딕셔너리를 유지하고 별도의 함수를 선언
from typing import TypedDict

class BookDict(TypedDict):
    isbn: str
    title: str
    authors: list[str]
    pagecount: int

# 메서드 대신 함수를 정의
def get_book_summary(book: BookDict) -> str:
    return f"{book['title']} by {', '.join(book['authors'])}"

pp: BookDict = {
    'title': 'Programming Pearls',
    'authors': ['Jon Bentley'],
    'isbn': '0201657880',
    'pagecount': 256
}

print(get_book_summary(pp))


Programming Pearls by Jon Bentley


In [13]:
# 데이터와 기능을 함께 묶고 싶은 경우 → dataclass 사용 (권장)

from dataclasses import dataclass

@dataclass
class Book:
    isbn: str
    title: str
    authors: list[str]
    pagecount: int

    # 메서드 선언 가능!
    def summary(self):
        return f"{self.title} by {', '.join(self.authors)}"

# 사용 예시
pp = Book(
    title='Programming Pearls',
    authors=['Jon Bentley'],
    isbn='0201657880',
    pagecount=256
)

print(pp.summary())  # 출력: Programming Pearls by Jon Bentley


Programming Pearls by Jon Bentley
